In [1]:
import os
import pandas as pd
from HtImageFormater import HtImageFormater
import numpy as np
import tifffile as tiff
import imagecodecs
import ipywidgets
from tqdm.auto import tqdm


In [2]:
image_formatter = HtImageFormater()
image_dir = r"E:\Miao\1st 24 hours-20260616_CC10-ntc 2r2a2-H2B_003"
image_formatter.read_ScanR(image_dir)
output_dir = r"E:\Miao\1st 24 hours-20260616_CC10-ntc 2r2a2-H2B_003\Merged"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)


In [ ]:
image_data = image_formatter._image_data
image_data # type: ignore

In [ ]:
image_data["well_names"].unique().shape[0]

In [ ]:
for image_id in tqdm(image_formatter._image_data["image_id"].unique(), desc="Exporting images", unit="img"):
    print(f"{image_id}")

In [ ]:
from itertools import product
import logging
logging.getLogger("tifffile").setLevel(logging.ERROR)

image_id = 13
channels = None
z_indices = None
times = None
show_progress = False
image_formatter.validate_image_data()

grp = image_formatter._image_data[image_formatter._image_data["image_id"] == image_id].copy() # type: ignore

if channels is None:
    channels = grp["channel_number"].unique()
if z_indices is None:
    z_indices = grp["z-index"].unique()
if times is None:
    times = grp["time"].unique()

c_to_idx = {c: idx for idx, c in enumerate(channels)}
z_to_idx = {z: idx for idx, z in enumerate(z_indices)}
t_to_idx = {t: idx for idx, t in enumerate(times)}

first_img = tiff.imread(grp.iloc[0]["image_paths"])
y, x = first_img.shape  # type: ignore # image is (Y, X) 

final_img = np.zeros((y, x, len(channels), len(z_indices), len(times)), dtype=first_img.dtype) # type: ignore

channels_nz = [c for c in channels if c != 0] # Exclude the empty channel (0) from the loop.
total_iters = len(channels_nz) * len(z_indices) * len(times)

with tqdm(total=total_iters, desc="Building image...", unit="img", disable=not show_progress) as pbar:
    for channel, zstep, time in product(channels_nz, z_indices, times):
        print(f"Processing channel: {channel}, z-index: {zstep}, time: {time}")
        row = grp[
            (grp["channel_number"] == channel) &
            (grp["z-index"] == zstep) &
            (grp["time"] == time)
        ]

        try:
            img = tiff.imread(row["image_paths"].values[0])  # (Y, X)
        except Exception as e:
            print(f"Error reading image for combination well = {row['well_names'].values[0]}, site={row['site'].values[0]}, channel={channel}, z={zstep}, t={time}. File is likely corrupted. Skipping image. \nError: {e}")
            continue

        final_img[:, :,
            c_to_idx[row["channel_number"].values[0]],
            z_to_idx[row["z-index"].values[0]],
            t_to_idx[row["time"].values[0]]
            ] = img

        pbar.update(1)

final_img = np.squeeze(final_img)  # Remove singleton dimensions if any

In [ ]:
grp

In [3]:
image_formatter.export_images(output_dir=output_dir, channels=None)

Exporting images:   0%|          | 0/360 [00:00<?, ?img/s]

Error reading image for combination well = E4, site=13, channel=1, z=0, t=30. File is likely corrupted. Skipping image. 
Error: list index out of range
